# 01 Data Joining — Women's

This notebook joins all women's raw data sources into clean, unified datasets for downstream EDA, feature engineering, and modeling.

**Key differences from men's pipeline:**
- Compact results start in 1998 (vs. 1985 for men's)
- Detailed box score results start in 2010 (vs. 2003 for men's)
- No Massey Ordinals available — must derive team quality entirely from game results and box scores
- No coaches data available
- ~1.5% of games in 2010–2012 may be missing detailed results

**Inputs** (from `00_data_collection/`):
- `WRegularSeasonCompactResults.csv` — game scores, 1998–2026
- `WRegularSeasonDetailedResults.csv` — box scores, 2010–2026
- `WNCAATourneyCompactResults.csv` — tournament scores, 1998–2025
- `WNCAATourneyDetailedResults.csv` — tournament box scores, 2010–2025
- `WNCAATourneySeeds.csv` — tournament seeds
- `WTeams.csv`, `WTeamConferences.csv`, `Conferences.csv`, `WSeasons.csv`

**Outputs** (to S3 `01_data_joining/womens/`):
1. `regular_season_games.parquet` — all regular season games with detailed stats where available
2. `tourney_games.parquet` — all tournament games with detailed stats where available
3. `team_season_stats.parquet` — per-team per-season aggregated statistics
4. `tourney_seeds.parquet` — cleaned tournament seeds with numeric seed extracted
5. `team_metadata.parquet` — team info and conferences per season

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_csv(filename):
    """Read CSV from S3 if available, otherwise fall back to local."""
    try:
        return pd.read_csv(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_csv(f"{LOCAL_DATA}{filename}")

In [3]:
def aggregate_team_season(team_games_df):
    """
    Compute per-team per-season aggregate stats from team-centric game rows.
    """
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    opp_box_cols = [f'Opp{c}' for c in box_cols]
    
    agg_dict = {
        'Win': ['sum', 'count'],
        'Score': 'mean',
        'OppScore': 'mean',
    }
    
    for col in box_cols + opp_box_cols:
        if col in team_games_df.columns:
            agg_dict[col] = 'mean'
    
    stats = team_games_df.groupby(['Season', 'TeamID']).agg(agg_dict)
    
    # Flatten MultiIndex columns
    stats.columns = ['_'.join(col).strip('_') for col in stats.columns]
    stats = stats.rename(columns={
        'Win_sum': 'Wins',
        'Win_count': 'Games',
        'Score_mean': 'AvgScore',
        'OppScore_mean': 'AvgOppScore',
    })
    
    # Rename box score columns to Avg prefix
    for col in box_cols + opp_box_cols:
        if f'{col}_mean' in stats.columns:
            stats = stats.rename(columns={f'{col}_mean': f'Avg{col}'})
    
    stats['Losses'] = stats['Games'] - stats['Wins']
    stats['WinPct'] = stats['Wins'] / stats['Games']
    stats['AvgPointDiff'] = stats['AvgScore'] - stats['AvgOppScore']
    
    # Derived shooting percentages and efficiency (only for seasons with detailed stats)
    if 'AvgFGM' in stats.columns:
        stats['FGPct'] = stats['AvgFGM'] / stats['AvgFGA']
        stats['FG3Pct'] = stats['AvgFGM3'] / stats['AvgFGA3']
        stats['FTPct'] = stats['AvgFTM'] / stats['AvgFTA']
        stats['OppFGPct'] = stats['AvgOppFGM'] / stats['AvgOppFGA']
        stats['OppFG3Pct'] = stats['AvgOppFGM3'] / stats['AvgOppFGA3']
        
        # Possessions estimate: FGA - OR + TO + 0.475 * FTA
        stats['AvgPoss'] = (stats['AvgFGA'] - stats['AvgOR'] + stats['AvgTO'] + 0.475 * stats['AvgFTA'])
        stats['AvgOppPoss'] = (stats['AvgOppFGA'] - stats['AvgOppOR'] + stats['AvgOppTO'] + 0.475 * stats['AvgOppFTA'])
        
        # Offensive and Defensive Efficiency (points per possession)
        stats['OffEff'] = stats['AvgScore'] / stats['AvgPoss']
        stats['DefEff'] = stats['AvgOppScore'] / stats['AvgOppPoss']
        stats['NetEff'] = stats['OffEff'] - stats['DefEff']
        
        # --- Margin features (team stat - opponent stat) ---
        stats['FGM_Margin'] = stats['AvgFGM'] - stats['AvgOppFGM']
        stats['FGA_Margin'] = stats['AvgFGA'] - stats['AvgOppFGA']
        stats['FGM3_Margin'] = stats['AvgFGM3'] - stats['AvgOppFGM3']
        stats['FTM_Margin'] = stats['AvgFTM'] - stats['AvgOppFTM']
        stats['FTA_Margin'] = stats['AvgFTA'] - stats['AvgOppFTA']
        stats['OR_Margin'] = stats['AvgOR'] - stats['AvgOppOR']
        stats['DR_Margin'] = stats['AvgDR'] - stats['AvgOppDR']
        stats['DRB_Pct'] = stats['AvgDR'] / (stats['AvgDR'] + stats['AvgOppOR'])
        stats['TotalReb_Margin'] = (stats['AvgOR'] + stats['AvgDR']) - (stats['AvgOppOR'] + stats['AvgOppDR'])
        stats['TO_Margin'] = stats['AvgOppTO'] - stats['AvgTO']  # flipped: opponent TOs are good
        stats['Stl_Margin'] = stats['AvgStl'] - stats['AvgOppStl']
        stats['Blk_Margin'] = stats['AvgBlk'] - stats['AvgOppBlk']
        stats['Ast_Margin'] = stats['AvgAst'] - stats['AvgOppAst']
        
        # --- Advanced shooting metrics ---
        stats['eFGPct'] = (stats['AvgFGM'] + 0.5 * stats['AvgFGM3']) / stats['AvgFGA']
        stats['TSPct'] = stats['AvgScore'] / (2 * (stats['AvgFGA'] + 0.44 * stats['AvgFTA']))
        stats['FTr'] = stats['AvgFTA'] / stats['AvgFGA']  # free throw rate
        stats['ThreePAr'] = stats['AvgFGA3'] / stats['AvgFGA']  # 3-point attempt rate
        
        # Opponent advanced shooting
        stats['OppeFGPct'] = (stats['AvgOppFGM'] + 0.5 * stats['AvgOppFGM3']) / stats['AvgOppFGA']
        stats['OppTSPct'] = stats['AvgOppScore'] / (2 * (stats['AvgOppFGA'] + 0.44 * stats['AvgOppFTA']))
    
    stats = stats.reset_index()
    return stats

In [4]:
def compute_momentum_features(team_games_df):
    """
    Compute weighted win percentage, winning streak, and losing streak for each team-season.
    - WeightedWinPct: weighted average of W/L where weights increase linearly
      with game order (more recent games weighted higher)
    - WinStreak: number of consecutive wins going into the tournament
    - LossStreak: number of consecutive losses going into the tournament
    """
    df = team_games_df.sort_values(['Season', 'TeamID', 'DayNum']).copy()
    
    results = []
    for (season, team), group in df.groupby(['Season', 'TeamID']):
        wins = group['Win'].values  # array of 1s and 0s
        n = len(wins)
        
        # Weighted win pct: weights = [1, 2, ..., n]
        weights = np.arange(1, n + 1, dtype=float)
        weighted_win_pct = np.average(wins, weights=weights)
        
        # Win streak: count consecutive wins from the end
        win_streak = 0
        for w in reversed(wins):
            if w == 1:
                win_streak += 1
            else:
                break
        
        # Loss streak: count consecutive losses from the end
        loss_streak = 0
        for w in reversed(wins):
            if w == 0:
                loss_streak += 1
            else:
                break
        
        results.append({
            'Season': season,
            'TeamID': team,
            'WeightedWinPct': weighted_win_pct,
            'WinStreak': win_streak,
            'LossStreak': loss_streak,
        })
    
    return pd.DataFrame(results)

#### Constants

In [5]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "01_data_joining"
INPUT_PREFIX = f"s3://{BUCKET}/00_data_collection/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

# For local development, fall back to local data directory
LOCAL_DATA = "../00_data_collection/"
LOCAL_OUTPUT = "output/"

#### Make output directory

In [6]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Raw Data

In [7]:
# Game results
reg_compact = read_csv("WRegularSeasonCompactResults.csv")
reg_detailed = read_csv("WRegularSeasonDetailedResults.csv")
tourney_compact = read_csv("WNCAATourneyCompactResults.csv")
tourney_detailed = read_csv("WNCAATourneyDetailedResults.csv")

# Tournament metadata
seeds = read_csv("WNCAATourneySeeds.csv")
seasons = read_csv("WSeasons.csv")

# Team metadata (no coaches file for women's)
teams = read_csv("WTeams.csv")
team_conferences = read_csv("WTeamConferences.csv")
conferences = read_csv("Conferences.csv")

print(f"Regular season compact: {reg_compact.shape}")
print(f"Regular season detailed: {reg_detailed.shape}")
print(f"Tourney compact: {tourney_compact.shape}")
print(f"Tourney detailed: {tourney_detailed.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Teams: {teams.shape}")

Regular season compact: (142507, 8)
Regular season detailed: (87187, 34)
Tourney compact: (1717, 8)
Tourney detailed: (961, 34)
Seeds: (1812, 3)
Teams: (379, 2)


## 2. Join Regular Season Games

Merge compact (1998–2026) and detailed (2010–2026) results. Pre-2010 seasons will have NaN for box score columns. Note that ~1.5% of 2010–2012 games may also be missing detailed results.

In [8]:
compact_cols = reg_compact.columns.tolist()
detail_only_cols = [c for c in reg_detailed.columns if c not in compact_cols]

reg_games = reg_compact.merge(
    reg_detailed[compact_cols + detail_only_cols],
    on=compact_cols,
    how='left'
)

print(f"Regular season games: {reg_games.shape}")
print(f"Seasons with detailed stats: {reg_games.dropna(subset=['WFGM']).Season.nunique()}")
print(f"Seasons without detailed stats: {reg_games[reg_games['WFGM'].isna()].Season.nunique()}")

# Check for missing detailed results in 2010-2012 range
early_detail = reg_games[(reg_games.Season >= 2010) & (reg_games.Season <= 2012)]
missing_pct = early_detail['WFGM'].isna().mean() * 100
print(f"\nMissing detailed stats in 2010-2012: {missing_pct:.1f}%")

Regular season games: (142507, 34)
Seasons with detailed stats: 17
Seasons without detailed stats: 15

Missing detailed stats in 2010-2012: 1.4%


## 3. Join Tournament Games

In [9]:
tourney_games = tourney_compact.merge(
    tourney_detailed[[c for c in tourney_detailed.columns]],
    on=compact_cols,
    how='left'
)

print(f"Tournament games: {tourney_games.shape}")
print(f"Tournament seasons: {tourney_games.Season.min()} - {tourney_games.Season.max()}")

Tournament games: (1717, 34)
Tournament seasons: 1998 - 2025


## 4. Build Team-Season Aggregated Statistics

Convert winner/loser format to team-centric rows, then aggregate per team per season. Since there are no Massey Ordinals for women's, these derived stats are especially important as the primary signal for team quality.

In [10]:
def build_team_game_rows(games_df):
    """
    Convert winner/loser format into team-centric rows.
    Each game produces two rows: one for each team.
    """
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    has_detail = 'WFGM' in games_df.columns and games_df['WFGM'].notna().any()
    
    # Winner rows
    w = games_df[['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']].copy()
    w.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    w['Win'] = 1
    w['Loc'] = w['WLoc'].map({'H': 'H', 'A': 'A', 'N': 'N'})
    
    # Loser rows
    l = games_df[['Season', 'DayNum', 'LTeamID', 'LScore', 'WTeamID', 'WScore', 'WLoc', 'NumOT']].copy()
    l.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    l['Win'] = 0
    l['Loc'] = l['WLoc'].map({'H': 'A', 'A': 'H', 'N': 'N'})
    
    if has_detail:
        for col in box_cols:
            w[col] = games_df[f'W{col}'].values
            w[f'Opp{col}'] = games_df[f'L{col}'].values
            l[col] = games_df[f'L{col}'].values
            l[f'Opp{col}'] = games_df[f'W{col}'].values
    
    w = w.drop(columns=['WLoc'])
    l = l.drop(columns=['WLoc'])
    
    return pd.concat([w, l], ignore_index=True)

team_games = build_team_game_rows(reg_games)
print(f"Team-game rows: {team_games.shape}")
team_games.head()

Team-game rows: (285014, 35)


,Season,DayNum,TeamID,Score,OppID,OppScore,NumOT,Win,Loc,FGM,...,Ast,OppAst,TO,OppTO,Stl,OppStl,Blk,OppBlk,PF,OppPF
0,1998,18,3104,91,3202,41,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998,18,3163,87,3221,76,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998,18,3222,66,3261,59,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1998,18,3307,69,3365,62,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1998,18,3349,115,3411,35,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
team_season_stats = aggregate_team_season(team_games)
print(f"Team-season stats: {team_season_stats.shape}")
print(f"Columns: {team_season_stats.columns.tolist()}")

# Show how many team-seasons have efficiency stats vs. only basic stats
has_eff = team_season_stats['OffEff'].notna().sum() if 'OffEff' in team_season_stats.columns else 0
print(f"\nTeam-seasons with efficiency stats: {has_eff} / {len(team_season_stats)}")
team_season_stats.head()

Team-season stats: (9851, 64)
Columns: ['Season', 'TeamID', 'Wins', 'Games', 'AvgScore', 'AvgOppScore', 'AvgFGM', 'AvgFGA', 'AvgFGM3', 'AvgFGA3', 'AvgFTM', 'AvgFTA', 'AvgOR', 'AvgDR', 'AvgAst', 'AvgTO', 'AvgStl', 'AvgBlk', 'AvgPF', 'AvgOppFGM', 'AvgOppFGA', 'AvgOppFGM3', 'AvgOppFGA3', 'AvgOppFTM', 'AvgOppFTA', 'AvgOppOR', 'AvgOppDR', 'AvgOppAst', 'AvgOppTO', 'AvgOppStl', 'AvgOppBlk', 'AvgOppPF', 'Losses', 'WinPct', 'AvgPointDiff', 'FGPct', 'FG3Pct', 'FTPct', 'OppFGPct', 'OppFG3Pct', 'AvgPoss', 'AvgOppPoss', 'OffEff', 'DefEff', 'NetEff', 'FGM_Margin', 'FGA_Margin', 'FGM3_Margin', 'FTM_Margin', 'FTA_Margin', 'OR_Margin', 'DR_Margin', 'DRB_Pct', 'TotalReb_Margin', 'TO_Margin', 'Stl_Margin', 'Blk_Margin', 'Ast_Margin', 'eFGPct', 'TSPct', 'FTr', 'ThreePAr', 'OppeFGPct', 'OppTSPct']

Team-seasons with efficiency stats: 5965 / 9851


,Season,TeamID,Wins,Games,AvgScore,AvgOppScore,AvgFGM,AvgFGA,AvgFGM3,AvgFGA3,...,TO_Margin,Stl_Margin,Blk_Margin,Ast_Margin,eFGPct,TSPct,FTr,ThreePAr,OppeFGPct,OppTSPct
0,1998,3102,4,24,57.291667,77.916667,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998,3103,11,29,69.241379,75.103448,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998,3104,21,30,76.566667,63.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1998,3106,6,21,61.238095,69.190476,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1998,3108,12,23,67.826087,66.521739,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# --- Strength of Schedule (SOS): average opponent WinPct ---
sos_data = team_games[['Season', 'TeamID', 'OppID']].merge(
    team_season_stats[['Season', 'TeamID', 'WinPct']].rename(
        columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'}
    ),
    on=['Season', 'OppID'],
    how='left'
)
sos = sos_data.groupby(['Season', 'TeamID'])['OppWinPct'].mean().reset_index()
sos.columns = ['Season', 'TeamID', 'SOS']

# --- Score Volatility: std dev of point differential ---
team_games_temp = team_games.copy()
team_games_temp['PointDiff'] = team_games_temp['Score'] - team_games_temp['OppScore']
score_std = team_games_temp.groupby(['Season', 'TeamID'])['PointDiff'].std().reset_index()
score_std.columns = ['Season', 'TeamID', 'ScoreStd']

# --- Quality Wins: win% against opponents with WinPct >= 0.6 ---
quality_data = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    team_season_stats[['Season', 'TeamID', 'WinPct']].rename(
        columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'}
    ),
    on=['Season', 'OppID'],
    how='left'
)
quality_games = quality_data[quality_data['OppWinPct'] >= 0.6]
quality_wins = quality_games.groupby(['Season', 'TeamID']).agg(
    QualityWinPct=('Win', 'mean'),
    QualityGames=('Win', 'count')
).reset_index()

# --- Conference Tournament Champion ---
conf_tourney = read_csv("WConferenceTourneyGames.csv")
conf_champ = (
    conf_tourney
    .sort_values('DayNum')
    .groupby(['Season', 'ConfAbbrev'])
    .last()
    .reset_index()
)
conf_champ = conf_champ[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'})
conf_champ['ConfTourneyChamp'] = 1

# --- Conference Regular Season Champion ---
conf_membership = team_conferences[['Season', 'TeamID', 'ConfAbbrev']]

conf_games = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    conf_membership, on=['Season', 'TeamID'], how='left'
).rename(columns={'ConfAbbrev': 'TeamConf'})
conf_games = conf_games.merge(
    conf_membership.rename(columns={'TeamID': 'OppID', 'ConfAbbrev': 'OppConf'}),
    on=['Season', 'OppID'], how='left'
)

conf_only = conf_games[conf_games['TeamConf'] == conf_games['OppConf']].copy()

conf_record = conf_only.groupby(['Season', 'TeamID', 'TeamConf']).agg(
    ConfWins=('Win', 'sum'),
    ConfGames=('Win', 'count')
).reset_index()
conf_record['ConfWinPct'] = conf_record['ConfWins'] / conf_record['ConfGames']

best_conf_pct = conf_record.groupby(['Season', 'TeamConf'])['ConfWinPct'].max().reset_index()
best_conf_pct.columns = ['Season', 'TeamConf', 'BestConfWinPct']

conf_record = conf_record.merge(best_conf_pct, on=['Season', 'TeamConf'], how='left')
conf_reg_champ = conf_record[conf_record['ConfWinPct'] == conf_record['BestConfWinPct']][['Season', 'TeamID']].copy()
conf_reg_champ['ConfRegSeasonChamp'] = 1

# --- Power Conference Win Pct ---
power_conf_abbrevs_set = {'acc', 'big_ten', 'big_twelve', 'sec', 'pac_twelve', 'big_east'}

power_conf_data = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    conf_membership.rename(columns={'TeamID': 'OppID', 'ConfAbbrev': 'OppConf'}),
    on=['Season', 'OppID'], how='left'
)
power_conf_games = power_conf_data[power_conf_data['OppConf'].isin(power_conf_abbrevs_set)]
power_conf_wins = power_conf_games.groupby(['Season', 'TeamID']).agg(
    PowerConfWinPct=('Win', 'mean'),
    PowerConfGames=('Win', 'count')
).reset_index()

# --- Momentum features ---
momentum = compute_momentum_features(team_games)

# --- Merge all into team_season_stats ---
team_season_stats = team_season_stats.merge(sos, on=['Season', 'TeamID'], how='left')
team_season_stats = team_season_stats.merge(score_std, on=['Season', 'TeamID'], how='left')
team_season_stats = team_season_stats.merge(
    quality_wins[['Season', 'TeamID', 'QualityWinPct']],
    on=['Season', 'TeamID'], how='left'
)
team_season_stats['QualityWinPct'] = team_season_stats['QualityWinPct'].fillna(0)
team_season_stats = team_season_stats.merge(conf_champ, on=['Season', 'TeamID'], how='left')
team_season_stats['ConfTourneyChamp'] = team_season_stats['ConfTourneyChamp'].fillna(0).astype(int)
team_season_stats = team_season_stats.merge(conf_reg_champ, on=['Season', 'TeamID'], how='left')
team_season_stats['ConfRegSeasonChamp'] = team_season_stats['ConfRegSeasonChamp'].fillna(0).astype(int)
team_season_stats = team_season_stats.merge(
    power_conf_wins[['Season', 'TeamID', 'PowerConfWinPct']],
    on=['Season', 'TeamID'], how='left'
)
team_season_stats['PowerConfWinPct'] = team_season_stats['PowerConfWinPct'].fillna(0)
team_season_stats = team_season_stats.merge(momentum, on=['Season', 'TeamID'], how='left')

print(f"Team-season stats after new features: {team_season_stats.shape}")
print(f"  SOS range: [{team_season_stats['SOS'].min():.3f}, {team_season_stats['SOS'].max():.3f}]")
print(f"  ScoreStd range: [{team_season_stats['ScoreStd'].min():.1f}, {team_season_stats['ScoreStd'].max():.1f}]")
print(f"  QualityWinPct mean: {team_season_stats['QualityWinPct'].mean():.3f}")
print(f"  ConfTourneyChamp count: {team_season_stats['ConfTourneyChamp'].sum()}")
print(f"  ConfRegSeasonChamp count: {team_season_stats['ConfRegSeasonChamp'].sum()}")
print(f"  PowerConfWinPct mean: {team_season_stats['PowerConfWinPct'].mean():.3f}")
print(f"  WeightedWinPct mean: {team_season_stats['WeightedWinPct'].mean():.3f}")
print(f"  WinStreak mean: {team_season_stats['WinStreak'].mean():.1f}")
print(f"  LossStreak mean: {team_season_stats['LossStreak'].mean():.1f}")

In [12]:
# --- Strength of Schedule (SOS): average opponent WinPct ---
sos_data = team_games[['Season', 'TeamID', 'OppID']].merge(
    team_season_stats[['Season', 'TeamID', 'WinPct']].rename(
        columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'}
    ),
    on=['Season', 'OppID'],
    how='left'
)
sos = sos_data.groupby(['Season', 'TeamID'])['OppWinPct'].mean().reset_index()
sos.columns = ['Season', 'TeamID', 'SOS']

# --- Score Volatility: std dev of point differential ---
team_games_temp = team_games.copy()
team_games_temp['PointDiff'] = team_games_temp['Score'] - team_games_temp['OppScore']
score_std = team_games_temp.groupby(['Season', 'TeamID'])['PointDiff'].std().reset_index()
score_std.columns = ['Season', 'TeamID', 'ScoreStd']

# --- Quality Wins: win% against opponents with WinPct >= 0.6 ---
quality_data = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    team_season_stats[['Season', 'TeamID', 'WinPct']].rename(
        columns={'TeamID': 'OppID', 'WinPct': 'OppWinPct'}
    ),
    on=['Season', 'OppID'],
    how='left'
)
quality_games = quality_data[quality_data['OppWinPct'] >= 0.6]
quality_wins = quality_games.groupby(['Season', 'TeamID']).agg(
    QualityWinPct=('Win', 'mean'),
    QualityGames=('Win', 'count')
).reset_index()

# --- Conference Tournament Champion ---
conf_tourney = read_csv("WConferenceTourneyGames.csv")
conf_champ = (
    conf_tourney
    .sort_values('DayNum')
    .groupby(['Season', 'ConfAbbrev'])
    .last()
    .reset_index()
)
conf_champ = conf_champ[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'})
conf_champ['ConfTourneyChamp'] = 1

# --- Conference Regular Season Champion ---
conf_membership = team_conferences[['Season', 'TeamID', 'ConfAbbrev']]

conf_games = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    conf_membership, on=['Season', 'TeamID'], how='left'
).rename(columns={'ConfAbbrev': 'TeamConf'})
conf_games = conf_games.merge(
    conf_membership.rename(columns={'TeamID': 'OppID', 'ConfAbbrev': 'OppConf'}),
    on=['Season', 'OppID'], how='left'
)

conf_only = conf_games[conf_games['TeamConf'] == conf_games['OppConf']].copy()

conf_record = conf_only.groupby(['Season', 'TeamID', 'TeamConf']).agg(
    ConfWins=('Win', 'sum'),
    ConfGames=('Win', 'count')
).reset_index()
conf_record['ConfWinPct'] = conf_record['ConfWins'] / conf_record['ConfGames']

best_conf_pct = conf_record.groupby(['Season', 'TeamConf'])['ConfWinPct'].max().reset_index()
best_conf_pct.columns = ['Season', 'TeamConf', 'BestConfWinPct']

conf_record = conf_record.merge(best_conf_pct, on=['Season', 'TeamConf'], how='left')
conf_reg_champ = conf_record[conf_record['ConfWinPct'] == conf_record['BestConfWinPct']][['Season', 'TeamID']].copy()
conf_reg_champ['ConfRegSeasonChamp'] = 1

# --- Power Conference Win Pct ---
power_conf_abbrevs_set = {'acc', 'big_ten', 'big_twelve', 'sec', 'pac_twelve', 'big_east'}

power_conf_data = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    conf_membership.rename(columns={'TeamID': 'OppID', 'ConfAbbrev': 'OppConf'}),
    on=['Season', 'OppID'], how='left'
)
power_conf_games = power_conf_data[power_conf_data['OppConf'].isin(power_conf_abbrevs_set)]
power_conf_wins = power_conf_games.groupby(['Season', 'TeamID']).agg(
    PowerConfWinPct=('Win', 'mean'),
    PowerConfGames=('Win', 'count')
).reset_index()

# --- Momentum features ---
momentum = compute_momentum_features(team_games)

# --- Merge all into team_season_stats ---
team_season_stats = team_season_stats.merge(sos, on=['Season', 'TeamID'], how='left')
team_season_stats = team_season_stats.merge(score_std, on=['Season', 'TeamID'], how='left')
team_season_stats = team_season_stats.merge(
    quality_wins[['Season', 'TeamID', 'QualityWinPct']],
    on=['Season', 'TeamID'], how='left'
)
team_season_stats['QualityWinPct'] = team_season_stats['QualityWinPct'].fillna(0)
team_season_stats = team_season_stats.merge(conf_champ, on=['Season', 'TeamID'], how='left')
team_season_stats['ConfTourneyChamp'] = team_season_stats['ConfTourneyChamp'].fillna(0).astype(int)
team_season_stats = team_season_stats.merge(conf_reg_champ, on=['Season', 'TeamID'], how='left')
team_season_stats['ConfRegSeasonChamp'] = team_season_stats['ConfRegSeasonChamp'].fillna(0).astype(int)
team_season_stats = team_season_stats.merge(
    power_conf_wins[['Season', 'TeamID', 'PowerConfWinPct']],
    on=['Season', 'TeamID'], how='left'
)
team_season_stats['PowerConfWinPct'] = team_season_stats['PowerConfWinPct'].fillna(0)
team_season_stats = team_season_stats.merge(momentum, on=['Season', 'TeamID'], how='left')

print(f"Team-season stats after new features: {team_season_stats.shape}")
print(f"  SOS range: [{team_season_stats['SOS'].min():.3f}, {team_season_stats['SOS'].max():.3f}]")
print(f"  ScoreStd range: [{team_season_stats['ScoreStd'].min():.1f}, {team_season_stats['ScoreStd'].max():.1f}]")
print(f"  QualityWinPct mean: {team_season_stats['QualityWinPct'].mean():.3f}")
print(f"  ConfTourneyChamp count: {team_season_stats['ConfTourneyChamp'].sum()}")
print(f"  ConfRegSeasonChamp count: {team_season_stats['ConfRegSeasonChamp'].sum()}")
print(f"  PowerConfWinPct mean: {team_season_stats['PowerConfWinPct'].mean():.3f}")
print(f"  WeightedWinPct mean: {team_season_stats['WeightedWinPct'].mean():.3f}")
print(f"  WinStreak mean: {team_season_stats['WinStreak'].mean():.1f}")
print(f"  LossStreak mean: {team_season_stats['LossStreak'].mean():.1f}")

Team-season stats after new features: (9851, 73)
  SOS range: [0.319, 0.691]
  ScoreStd range: [7.8, 30.8]
  QualityWinPct mean: 0.243
  ConfTourneyChamp count: 763
  ConfRegSeasonChamp count: 988
  PowerConfWinPct mean: 0.190
  WeightedWinPct mean: 0.491
  WinStreak mean: 0.9
  LossStreak mean: 2.0


In [13]:
# --- Home / Road Win Pct ---
for loc, label in [('H', 'Home'), ('A', 'Road')]:
    loc_games = team_games[team_games['Loc'] == loc]
    loc_stats = loc_games.groupby(['Season', 'TeamID']).agg(
        WinPct=('Win', 'mean'),
        Games=('Win', 'count')
    ).reset_index().rename(columns={'WinPct': f'{label}WinPct'})
    
    team_season_stats = team_season_stats.merge(
        loc_stats[['Season', 'TeamID', f'{label}WinPct']],
        on=['Season', 'TeamID'], how='left'
    )
    team_season_stats[f'{label}WinPct'] = team_season_stats[f'{label}WinPct'].fillna(0)
    
    n_with = (team_season_stats[f'{label}WinPct'] > 0).sum()
    print(f"  {label}WinPct: {n_with} team-seasons, mean={team_season_stats[f'{label}WinPct'].mean():.3f}")

  HomeWinPct: 9741 team-seasons, mean=0.598
  RoadWinPct: 9522 team-seasons, mean=0.394


## 5. Clean Tournament Seeds

Extract numeric seed from the seed string (e.g., `W01` → 1, `X16a` → 16).

In [14]:
tourney_seeds = seeds.copy()
tourney_seeds['Region'] = tourney_seeds['Seed'].str[0]
tourney_seeds['SeedNum'] = tourney_seeds['Seed'].str[1:3].astype(int)
tourney_seeds['PlayIn'] = tourney_seeds['Seed'].str[3:]  # 'a', 'b', or ''

print(f"Tournament seeds: {tourney_seeds.shape}")
print(f"Seed range: {tourney_seeds.SeedNum.min()} - {tourney_seeds.SeedNum.max()}")
print(f"Seasons: {tourney_seeds.Season.min()} - {tourney_seeds.Season.max()}")
tourney_seeds.head()

Tournament seeds: (1812, 6)
Seed range: 1 - 16
Seasons: 1998 - 2026


,Season,Seed,TeamID,Region,SeedNum,PlayIn
0,1998,W01,3330,W,1,
1,1998,W02,3163,W,2,
2,1998,W03,3112,W,3,
3,1998,W04,3301,W,4,
4,1998,W05,3272,W,5,


## 6. Build Team Metadata

Join team names and conference affiliations per season. No coaches data is available for women's.

In [15]:
# Join conferences with their descriptions
team_conf = team_conferences.merge(conferences, on='ConfAbbrev', how='left')
team_conf = team_conf.rename(columns={'Description': 'Conference'})

# Identify power conferences
power_conf_abbrevs = ['acc', 'big_ten', 'big_twelve', 'sec', 'pac_twelve', 'big_east']
team_conf['IsPowerConf'] = team_conf['ConfAbbrev'].isin(power_conf_abbrevs).astype(int)

# Join team names
team_meta = team_conf.merge(
    teams[['TeamID', 'TeamName']],
    on='TeamID',
    how='left'
)

print(f"Team metadata: {team_meta.shape}")
team_meta.head()

Team metadata: (9853, 6)


,Season,TeamID,ConfAbbrev,Conference,IsPowerConf,TeamName
0,1998,3102,wac,Western Athletic Conference,0,Air Force
1,1998,3103,mac,Mid-American Conference,0,Akron
2,1998,3104,sec,Southeastern Conference,1,Alabama
3,1998,3106,swac,Southwest Athletic Conference,0,Alabama St
4,1998,3108,swac,Southwest Athletic Conference,0,Alcorn St


## 7. Compute Elo Ratings

In [16]:
def compute_elo_ratings(games_df, k=32, home_advantage=100, mean_elo=1500, reversion_factor=0.33):
    """
    Compute Elo ratings from game results.
    """
    elo = {}
    season_end_elo = {}
    seasons = sorted(games_df['Season'].unique())
    
    for season in seasons:
        for team in elo:
            elo[team] = elo[team] * (1 - reversion_factor) + mean_elo * reversion_factor
        
        season_games = games_df[games_df['Season'] == season].sort_values('DayNum')
        
        for _, game in season_games.iterrows():
            w_id = game['WTeamID']
            l_id = game['LTeamID']
            if w_id not in elo: elo[w_id] = mean_elo
            if l_id not in elo: elo[l_id] = mean_elo
            
            w_elo = elo[w_id]
            l_elo = elo[l_id]
            
            if game['WLoc'] == 'H':
                w_elo_adj = w_elo + home_advantage
                l_elo_adj = l_elo
            elif game['WLoc'] == 'A':
                w_elo_adj = w_elo
                l_elo_adj = l_elo + home_advantage
            else:
                w_elo_adj = w_elo
                l_elo_adj = l_elo
            
            w_expected = 1.0 / (1.0 + 10 ** ((l_elo_adj - w_elo_adj) / 400.0))
            margin = game['WScore'] - game['LScore']
            margin_mult = np.log(abs(margin) + 1) * 0.5 + 0.5
            k_adj = k * margin_mult
            elo[w_id] = w_elo + k_adj * (1 - w_expected)
            elo[l_id] = l_elo + k_adj * (0 - (1 - w_expected))
        
        reg_season = season_games[season_games['DayNum'] < 132]
        teams_in_season = set(reg_season['WTeamID'].unique()) | set(reg_season['LTeamID'].unique())
        for team in teams_in_season:
            if team in elo:
                season_end_elo[(season, team)] = elo[team]
    
    elo_df = pd.DataFrame([
        {'Season': s, 'TeamID': t, 'Elo': e}
        for (s, t), e in season_end_elo.items()
    ])
    return elo_df

elo_ratings = compute_elo_ratings(reg_compact)
print(f"Elo ratings: {elo_ratings.shape}")
print(f"Elo range: [{elo_ratings.Elo.min():.0f}, {elo_ratings.Elo.max():.0f}]")
elo_ratings.head()

Elo ratings: (9851, 3)
Elo range: [904, 2260]


,Season,TeamID,Elo
0,1998,3102,1194.445878
1,1998,3103,1407.707992
2,1998,3104,1839.676287
3,1998,3106,1293.212438
4,1998,3108,1523.902350


## 8. Compute Recent Form (Last 14 Games)

In [17]:
def compute_recent_form(games_df, n_games=14):
    """Compute average stats from the last N regular season games before tournament."""
    team_games = build_team_game_rows(games_df)
    team_games = team_games[team_games['DayNum'] < 132].copy()
    team_games = team_games.sort_values(['Season', 'TeamID', 'DayNum'])
    recent = team_games.groupby(['Season', 'TeamID']).tail(n_games)
    
    agg_cols = {'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'}
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    for col in box_cols:
        if col in recent.columns:
            agg_cols[col] = 'mean'
    
    form = recent.groupby(['Season', 'TeamID']).agg(agg_cols).reset_index()
    rename_map = {'Win': 'RecentWinPct', 'Score': 'RecentAvgScore', 'OppScore': 'RecentAvgOppScore'}
    for col in box_cols:
        if col in form.columns:
            rename_map[col] = f'Recent{col}'
    form = form.rename(columns=rename_map)
    
    form['RecentAvgPointDiff'] = form['RecentAvgScore'] - form['RecentAvgOppScore']
    if 'RecentFGM' in form.columns:
        form['RecentFGPct'] = form['RecentFGM'] / form['RecentFGA']
        form['RecentPoss'] = form['RecentFGA'] - form['RecentOR'] + form['RecentTO'] + 0.475 * form['RecentFTA']
        form['RecentOffEff'] = form['RecentAvgScore'] / form['RecentPoss']
        form['RecentDefEff'] = form['RecentAvgOppScore'] / form['RecentPoss']
        form['RecentNetEff'] = form['RecentOffEff'] - form['RecentDefEff']
    
    return form

recent_form = compute_recent_form(reg_games)
print(f"Recent form stats: {recent_form.shape}")
recent_form.head()

Recent form stats: (9851, 24)


,Season,TeamID,RecentWinPct,RecentAvgScore,RecentAvgOppScore,RecentFGM,RecentFGA,RecentFGM3,RecentFGA3,RecentFTM,...,RecentTO,RecentStl,RecentBlk,RecentPF,RecentAvgPointDiff,RecentFGPct,RecentPoss,RecentOffEff,RecentDefEff,RecentNetEff
0,1998,3102,0.000000,55.857143,83.071429,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-27.214286,NaN,NaN,NaN,NaN,NaN
1,1998,3103,0.285714,71.285714,78.785714,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-7.500000,NaN,NaN,NaN,NaN,NaN
2,1998,3104,0.785714,74.000000,61.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,13.000000,NaN,NaN,NaN,NaN,NaN
3,1998,3106,0.214286,58.857143,67.928571,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-9.071429,NaN,NaN,NaN,NaN,NaN
4,1998,3108,0.714286,71.071429,62.714286,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,8.357143,NaN,NaN,NaN,NaN,NaN


## 9. Build Synthetic Quality Rankings

Since women's data has no Massey Ordinals, we build a synthetic quality ranking using tournament seeds as a proxy target. Seeded teams have a known quality ordering (seed 1 = best, 16 = worst). We train a Ridge regression model to predict seed number from team stats, then apply it to ALL teams to get a continuous quality score — analogous to a Massey rank.

In [18]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler as RankScaler

def build_synthetic_rankings(team_stats, seeds):
    """
    Build a synthetic quality ranking for women's teams using seeds as training signal.
    Teams with seeds have a known quality ordering (seed 1 = best, 16 = worst).
    We train a model to predict seed number from team stats, then apply it to ALL teams
    to get a continuous quality score (like a synthetic Massey rank).
    """
    # Features to predict quality
    predictor_cols = ['WinPct', 'AvgPointDiff', 'AvgScore', 'AvgOppScore', 'Wins', 'Losses', 'Games']
    detail_cols = ['OffEff', 'DefEff', 'NetEff', 'FGPct', 'FG3Pct', 'FTPct',
                   'AvgTO', 'AvgStl', 'AvgBlk', 'AvgOR', 'AvgDR', 'AvgAst']
    predictor_cols += [c for c in detail_cols if c in team_stats.columns]
    
    # Merge team stats with seeds (only seeded teams have a "rank")
    merged = team_stats[['Season', 'TeamID'] + predictor_cols].merge(
        seeds[['Season', 'TeamID', 'SeedNum']],
        on=['Season', 'TeamID'],
        how='left'
    )
    
    # Training data: seeded teams (they have a known quality ordering)
    train_data = merged[merged['SeedNum'].notna()].copy()
    # All data: predict for everyone
    all_data = merged.copy()
    
    print(f"  Training on {len(train_data)} seeded team-seasons")
    
    # Prepare features
    X_train = train_data[predictor_cols].copy()
    X_all = all_data[predictor_cols].copy()
    
    # Fill NaN with column medians from training data
    for col in predictor_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_all[col] = X_all[col].fillna(median_val)
    
    # Scale
    scaler = RankScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_all_scaled = scaler.transform(X_all)
    
    # Train model to predict seed (lower = better)
    y_train = train_data['SeedNum'].values
    model = Ridge(alpha=10.0)
    model.fit(X_train_scaled, y_train)
    
    train_corr = np.corrcoef(y_train, model.predict(X_train_scaled))[0, 1]
    print(f"  Seed prediction — Train R²: {model.score(X_train_scaled, y_train):.3f}, Correlation: {train_corr:.3f}")
    
    # Predict synthetic rank for ALL teams (not just seeded ones)
    # This gives a continuous quality score where lower = better (like Massey)
    all_data['SyntheticRank'] = model.predict(X_all_scaled)
    
    # For non-seeded teams, the predicted "seed" will be > 16, which is correct
    # (they're worse than tournament teams)
    print(f"  Synthetic rank range: [{all_data['SyntheticRank'].min():.1f}, {all_data['SyntheticRank'].max():.1f}]")
    print(f"  Mean for seeded teams: {all_data[all_data['SeedNum'].notna()]['SyntheticRank'].mean():.1f}")
    print(f"  Mean for non-seeded teams: {all_data[all_data['SeedNum'].isna()]['SyntheticRank'].mean():.1f}")
    
    return all_data[['Season', 'TeamID', 'SyntheticRank']]

synthetic_rankings = build_synthetic_rankings(team_season_stats, tourney_seeds)
print(f"\nSynthetic rankings: {synthetic_rankings.shape}")
synthetic_rankings.head()

  Training on 1812 seeded team-seasons
  Seed prediction — Train R²: 0.423, Correlation: 0.651
  Synthetic rank range: [-4.6, 28.3]
  Mean for seeded teams: 8.6
  Mean for non-seeded teams: 14.0

Synthetic rankings: (9851, 3)


,Season,TeamID,SyntheticRank
0,1998,3102,21.299224
1,1998,3103,15.051305
2,1998,3104,6.869679
3,1998,3106,16.097708
4,1998,3108,12.716144


## 9b. Win Pct vs Top-Ranked Opponents

Compute win percentage against top 5, top 10, and top 25 ranked opponents using SyntheticRank (women's equivalent of Massey Ordinals).

In [19]:
# Get opponent rankings (SyntheticRank) per season — lower = better
opp_ranks = synthetic_rankings[['Season', 'TeamID', 'SyntheticRank']].rename(
    columns={'TeamID': 'OppID', 'SyntheticRank': 'OppRank'}
)

# Join opponent rank onto team game rows
ranked_games = team_games[['Season', 'TeamID', 'OppID', 'Win']].merge(
    opp_ranks, on=['Season', 'OppID'], how='left'
)

# Compute win% against top N opponents for each threshold
for n, col_name in [(25, 'Top25WinPct'), (10, 'Top10WinPct'), (5, 'Top5WinPct')]:
    # For SyntheticRank, we need to find the Nth best rank per season
    # (lowest SyntheticRank values = best teams)
    season_thresholds = synthetic_rankings.groupby('Season')['SyntheticRank'].apply(
        lambda x: x.nsmallest(n).max()
    ).reset_index()
    season_thresholds.columns = ['Season', f'Top{n}Threshold']
    
    ranked_with_thresh = ranked_games.merge(season_thresholds, on='Season', how='left')
    top_n_games = ranked_with_thresh[ranked_with_thresh['OppRank'] <= ranked_with_thresh[f'Top{n}Threshold']]
    
    top_n_stats = top_n_games.groupby(['Season', 'TeamID']).agg(
        WinPct=('Win', 'mean'),
        Games=('Win', 'count')
    ).reset_index().rename(columns={'WinPct': col_name, 'Games': f'{col_name}_Games'})
    
    team_season_stats = team_season_stats.merge(
        top_n_stats[['Season', 'TeamID', col_name]],
        on=['Season', 'TeamID'], how='left'
    )
    team_season_stats[col_name] = team_season_stats[col_name].fillna(0)
    
    n_with_games = (team_season_stats[col_name] > 0).sum()
    print(f"  {col_name}: {n_with_games} team-seasons with games, mean={team_season_stats[col_name].mean():.3f}")

print(f"\nTeam-season stats after top-N features: {team_season_stats.shape}")

  Top25WinPct: 1986 team-seasons with games, mean=0.085
  Top10WinPct: 733 team-seasons with games, mean=0.037
  Top5WinPct: 313 team-seasons with games, mean=0.019

Team-season stats after top-N features: (9851, 78)


## 10. Save Outputs

Write all joined datasets to S3 (parquet format for efficiency).

In [20]:
outputs = {
    'regular_season_games': reg_games,
    'tourney_games': tourney_games,
    'team_season_stats': team_season_stats,
    'tourney_seeds': tourney_seeds,
    'team_metadata': team_meta,
    'elo_ratings': elo_ratings,
    'recent_form': recent_form,
    'synthetic_rankings': synthetic_rankings,
}

for name, df in outputs.items():
    # Save to S3
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/regular_season_games.parquet ((142507, 34))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/tourney_games.parquet ((1717, 34))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/team_season_stats.parquet ((9851, 78))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/tourney_seeds.parquet ((1812, 6))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/team_metadata.parquet ((9853, 6))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/elo_ratings.parquet ((9851, 3))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/recent_form.parquet ((9851, 24))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/womens/synthetic_rankings.parquet ((9851, 3))


## 11. Output Summary

In [21]:
print("=" * 60)
print("DATA JOINING SUMMARY — WOMEN'S")
print("=" * 60)
for name, df in outputs.items():
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()[:10]}{'...' if len(df.columns) > 10 else ''}")
    if 'Season' in df.columns:
        print(f"  Seasons: {df.Season.min()} - {df.Season.max()}")

print("\n" + "=" * 60)
print("NOTE: No Massey Ordinals available for women's.")
print("Synthetic quality rankings built using seeds as proxy target.")
print("Detailed box scores available from 2010 onwards only.")
print("=" * 60)

DATA JOINING SUMMARY — WOMEN'S

regular_season_games:
  Shape: (142507, 34)
  Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA']...
  Seasons: 1998 - 2026

tourney_games:
  Shape: (1717, 34)
  Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA']...
  Seasons: 1998 - 2025

team_season_stats:
  Shape: (9851, 78)
  Columns: ['Season', 'TeamID', 'Wins', 'Games', 'AvgScore', 'AvgOppScore', 'AvgFGM', 'AvgFGA', 'AvgFGM3', 'AvgFGA3']...
  Seasons: 1998 - 2026

tourney_seeds:
  Shape: (1812, 6)
  Columns: ['Season', 'Seed', 'TeamID', 'Region', 'SeedNum', 'PlayIn']
  Seasons: 1998 - 2026

team_metadata:
  Shape: (9853, 6)
  Columns: ['Season', 'TeamID', 'ConfAbbrev', 'Conference', 'IsPowerConf', 'TeamName']
  Seasons: 1998 - 2026

elo_ratings:
  Shape: (9851, 3)
  Columns: ['Season', 'TeamID', 'Elo']
  Seasons: 1998 - 2026

recent_form:
  Shape: (9851, 24)
  Columns: ['Season', 'TeamID', 'Re